In [1]:
import random
import string
from collections import defaultdict

# File Reader
def load_text(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as file:
            data = [line.strip() for line in file if line.strip()]
        return data
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        return []


# Text Preprocessing
def clean_and_split(sentence):

    cleaned = sentence.lower().translate(str.maketrans('', '', string.punctuation))
    return cleaned.split()

In [2]:
#  Model Training
def build_markov_model(lines):
    start_word_counts = defaultdict(int)
    one_step_counts = defaultdict(lambda: defaultdict(int))
    two_step_counts = defaultdict(lambda: defaultdict(int))

    for sentence in lines:
        words = clean_and_split(sentence)
        if not words:
            continue

        # Initial (start) word frequency
        start_word_counts[words[0]] += 1

        # First-order transitions (word -> next)
        for i in range(len(words) - 1):
            one_step_counts[words[i]][words[i + 1]] += 1

        # Second-order transitions ((w1, w2) -> next)
        for i in range(len(words) - 2):
            pair = (words[i], words[i + 1])
            two_step_counts[pair][words[i + 2]] += 1

    # Normalize counts into probabilities
    start_probs = normalize_single(start_word_counts)
    first_probs = normalize_nested_dict(one_step_counts)
    second_probs = normalize_nested_dict(two_step_counts)

    return start_probs, first_probs, second_probs


#  Normalization
def normalize_single(counter):
    total = sum(counter.values())
    return {word: count / total for word, count in counter.items()}


def normalize_nested_dict(counter_dict):
    normalized = {}
    for key, subdict in counter_dict.items():
        total = sum(subdict.values())
        normalized[key] = {word: val / total for word, val in subdict.items()}
    return normalized

In [3]:
#  Word Selection
def pick_word(probabilities):
    threshold = random.random()
    cumulative = 0
    for word, prob in probabilities.items():
        cumulative += prob
        if threshold <= cumulative:
            return word
    return random.choice(list(probabilities.keys()))


# Text Generation
def compose_poem(start_probs, first_probs, second_probs, lines=4, max_len=10):
    output = []

    for _ in range(lines):
        line = []
        w1 = pick_word(start_probs)
        line.append(w1)

        if w1 not in first_probs:
            output.append(' '.join(line))
            continue

        w2 = pick_word(first_probs[w1])
        line.append(w2)

        for _ in range(max_len - 2):
            pair = (line[-2], line[-1])
            if pair not in second_probs:
                break
            next_word = pick_word(second_probs[pair])
            line.append(next_word)

        output.append(' '.join(line))

    return '\n'.join(output)


In [4]:
# Main Program
if __name__ == "__main__":
    filename = "robert_frost.txt"
    text_lines = load_text(filename)

    if not text_lines:
        exit()

    init_p, first_p, second_p = build_markov_model(text_lines)
    generated = compose_poem(init_p, first_p, second_p)

    print("Generated Poem:\n")
    print(generated)

Generated Poem:

no other way to me these two
some word about the weather
today he said i could have done the thing i
the hitching posts
